# CASMI 2026 · RL77 Notebook 2 — Offline Candidate Store (Catalog, FP16 Memmap, FAISS)

*Assembled from **CASMI 2026 Colab Rubric.pdf** (pages 21–27). Python cells are the reviewed `experiments/runs/RL77/nb2/cell_NN.py` files; text cells are taken from the PDF.*

> **Assistant (end of the GLMR template):** Would you like the companion script to pre-compute and shard the PubChem/COCONUT ChemFormer embeddings into a memory-mapped FAISS index for your offline Kaggle dataset upload?
>
> **User prompt:** yes

This companion pipeline curates and deduplicates compounds from `train.parquet`, **COCONUT**, and **PubChem** by `inchikey14`, sorts the catalog by monoisotopic mass for $O(\log N)$ binary-search slicing, encodes molecules into `float16` memory-mapped **ChemFormer** embeddings alongside a **FAISS** index, and packages the artifacts for offline Kaggle execution.

## Prerequisites

| Input | Kaggle source | Provides |
| --- | --- | --- |
| Competition data | `enveda-CASMI26-molecule-id-mass-spectra` | `train.parquet`, `test.parquet`, `sample_submission.csv` |
| RL77 prerequisites | `scottweeden/casmi26-rl77-prereqs` | `rl77_env.py`, `rl77_weights.py`, `rl77_weights.json` (Evidently snapshot `rl77-v014`), upstream modules |
| Candidate structures | `scottweeden/casmi26-rl77-candidates` | `coconut_structures.parquet` (COCONUT natural products) |
| Notebook 1 outputs | `scottweeden/casmi26-rl77-nb1-glmr-outputs` | Two-stage GLMR checkpoint, retrieval library, validation split, submission |

- **Order:** Run notebooks 1 first and publish their outputs (`tools/kaggle_pipeline.py publish-output`).
- **Accelerator:** CPU (no GPU quota used); internet on for the test run so `ensure_packages()` can reach PyPI. Offline runs install from the wheels in notebook 2's output dataset.
- **Packages:** `rdkit==2026.03.3` (the metric's canonicalizer) and `faiss-cpu` are installed if the image lacks them; `torch`, `pyarrow`, `pandas`, `scipy` come from the Kaggle image.
- **Weights:** all blend weights, reward coefficients, and tolerances come from Evidently dataset `casmi-rl77-weights` (version `rl77-v014`, id `01a1ee49-89c9-7ce4-b327-df8b7cf1d48f`) through `W("...")`.
- **Outputs:** everything written to `/kaggle/working` is published as `scottweeden/casmi26-rl77-nb2-store-outputs` for the next notebook.

In [ ]:
# Prerequisites: locate the RL77 helper modules, install pinned packages, load the Evidently weights snapshot
import glob, os, sys
from pathlib import Path

_lib_hits = sorted(glob.glob("/kaggle/input/**/rl77_env.py", recursive=True))
if _lib_hits:
    RL77_LIB = Path(_lib_hits[0]).parent
elif os.environ.get("RL77_LIB_DIR"):
    RL77_LIB = Path(os.environ["RL77_LIB_DIR"])  # e.g. experiments/runs/RL77/build/datasets/casmi26-rl77-prereqs
else:
    raise FileNotFoundError("Attach the scottweeden/casmi26-rl77-prereqs dataset or set RL77_LIB_DIR.")
sys.path.insert(0, str(RL77_LIB))

import rl77_env as ENV
ENV.ensure_packages()                      # rdkit==2026.03.3 + faiss-cpu (offline wheels when attached)
from rl77_weights import load_rl77_weights
W = load_rl77_weights()                    # every blend weight / reward coefficient in this notebook reads W("...")
from rl77_nb1_glmr import *  # nb1: CASMI26 RL77 NB1 GLMR
ENV.log(f"Competition data: {ENV.competition_dir()} | outputs: {ENV.work_dir()}")


## Cell 1: Candidate Corpus Curation, `inchikey14` Deduplication & Mass Sorting

Sorting the deduplicated catalog by `exact_mass` enables contiguous memory-mapped slicing (`np.searchsorted`) during inference, avoiding full-database scans while supporting both the **Weight-based** and **Formula-based** retrieval libraries evaluated in GLMR.

> **Review fix:** Google Drive paths become the competition mount, the `casmi26-rl77-candidates` COCONUT export (495k structures), and `/kaggle/working`. Tautomer canonicalization of ~770k structures runs on a process pool under a time budget (tier 1 always completes). `exact_mass` is stored as float64 (float32 rounds ~0.06 ppm at 1000 Da), and an empty catalog raises instead of failing later in `sort_values`.

In [ ]:
# [Colab Companion Cell 1] Curate, canonicalize (RDKit 2026.03.3), deduplicate by InChIKey14, and sort by exact mass
import os
import json
import math
import multiprocessing as mp
from pathlib import Path
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors, inchi
from rdkit.Chem.MolStandardize import rdMolStandardize

RDLogger.DisableLog("rdApp.*")

tautomer_enumerator = rdMolStandardize.TautomerEnumerator()


class IndexCFG:
    RAW_TRAIN_PARQUET = ENV.competition_dir() / "train.parquet"
    # External natural-product structures: COCONUT export shipped as the casmi26-rl77-candidates dataset.
    EXTERNAL_STRUCTURES = ENV.dataset_dir(ENV.CANDIDATES_SLUG, "coconut_structures.parquet", required=False)
    EXPORT_DIR = ENV.work_dir()          # published as the casmi26-rl77-nb2-store-outputs dataset
    CATALOG_PARQUET = EXPORT_DIR / "candidate_catalog_sorted.parquet"
    EMBEDDINGS_MMAP = EXPORT_DIR / "chemformer_embs_fp16.mmap"
    FAISS_INDEX_PATH = EXPORT_DIR / "chemformer_ivfflat.index"
    FORMULA_INDEX_PATH = EXPORT_DIR / "formula_offsets.npz"

    MIN_MASS_DA = 100.0   # Cover test.parquet range (157 - 1159 Da) with margin
    MAX_MASS_DA = 1250.0
    D_MODEL = 256
    SHARD_BATCH_SIZE = 1024
    N_PROCS = max(1, os.cpu_count() or 1)
    STANDARDIZE_MINUTES = float(os.environ.get("RL77_STANDARDIZE_MINUTES", 70))


IndexCFG.EXPORT_DIR.mkdir(parents=True, exist_ok=True)


def standardize_compound(raw_smiles: str) -> Optional[Tuple[str, str, float, str]]:
    """
    Applies RDKit 2026.03.3 tautomer canonicalization and returns:
    (canonical_smiles, inchikey14, exact_monoisotopic_mass, molecular_formula)
    """
    if not raw_smiles or not isinstance(raw_smiles, str):
        return None
    mol = Chem.MolFromSmiles(raw_smiles.strip())
    if mol is None:
        return None
    try:
        # Keep largest covalent fragment if salt is present
        frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=True)
        if len(frags) > 1:
            mol = max(frags, key=lambda m: m.GetNumHeavyAtoms())

        canon_mol = tautomer_enumerator.Canonicalize(mol)
        exact_mass = float(Descriptors.ExactMolWt(canon_mol))
        if not (IndexCFG.MIN_MASS_DA <= exact_mass <= IndexCFG.MAX_MASS_DA):
            return None

        ik = inchi.MolToInchiKey(canon_mol)
        if not ik or len(ik) < 14:
            return None
        ik14 = ik[:14]

        canon_smi = Chem.MolToSmiles(canon_mol, isomericSmiles=False, canonical=True)
        formula = rdMolDescriptors.CalcMolFormula(canon_mol)
        return canon_smi, ik14, exact_mass, formula
    except Exception:
        return None


def standardize_many(smiles: List[str], deadline: Optional["ENV.Deadline"] = None) -> List[Optional[Tuple[str, str, float, str]]]:
    """Parallel standardize_compound over a process pool; stops submitting work at the deadline."""
    results: List[Optional[Tuple[str, str, float, str]]] = []
    chunk = 20_000
    with mp.get_context("fork").Pool(IndexCFG.N_PROCS) as pool:
        for start in range(0, len(smiles), chunk):
            if deadline is not None and deadline.expired():
                ENV.log(f"Standardization budget reached after {start:,} of {len(smiles):,} structures")
                break
            results.extend(pool.map(standardize_compound, smiles[start:start + chunk], chunksize=256))
    return results


def load_precomputed_standardization(path: Path) -> Optional[List[Tuple[str, str, float, str]]]:
    """Tier-2 results from tools/standardize_candidates.py (this cell's standardize_compound run on a local CPU
    pool), used only when they were produced with the RDKit version and mass range in this session."""
    import rdkit

    if not path.exists():
        return None
    meta = json.loads((pq.read_schema(path).metadata or {}).get(b"rl77", b"{}"))
    expected = {"rdkit": rdkit.__version__, "min_mass_da": IndexCFG.MIN_MASS_DA, "max_mass_da": IndexCFG.MAX_MASS_DA}
    if any(meta.get(k) != v for k, v in expected.items()):
        ENV.log(f"Ignoring {path.name}: built with {meta}, need {expected}")
        return None
    df = pd.read_parquet(path, columns=["canonical_smiles", "inchikey14", "exact_mass", "molecular_formula"])
    df = df.dropna(subset=["inchikey14"])
    ENV.log(f"Using {path.name}: {len(df):,} standardized structures (RDKit {meta['rdkit']}, {meta.get('minutes')} min locally)")
    return list(df.itertuples(index=False, name=None))


def build_mass_sorted_catalog(
    train_parquet_path: Path = IndexCFG.RAW_TRAIN_PARQUET,
    external_path: Optional[Path] = IndexCFG.EXTERNAL_STRUCTURES,
    output_parquet_path: Path = IndexCFG.CATALOG_PARQUET,
) -> pd.DataFrame:
    """
    Merges in-library training structures with external PubChem/COCONUT candidates,
    deduplicates strictly by inchikey14, and sorts ascending by exact_mass.
    """
    seen_ik14: Dict[str, Tuple[str, float, str, int]] = {}
    deadline = ENV.Deadline(IndexCFG.STANDARDIZE_MINUTES)

    # 1. Load unique structures from train.parquet (Tier 1: In-library priority)
    if train_parquet_path.exists():
        train_smi = pq.read_table(train_parquet_path, columns=["normalized_smiles"])["normalized_smiles"]
        unique_train_smi = [s for s in pd.unique(train_smi.to_pandas().dropna()) if isinstance(s, str)]
        for res in standardize_many(unique_train_smi):  # tier 1 always completes
            if res is not None:
                canon_smi, ik14, mass, formula = res
                if ik14 not in seen_ik14:
                    seen_ik14[ik14] = (canon_smi, mass, formula, 1)  # source=1 (train_lib)
        ENV.log(f"Tier 1 (train.parquet): {len(seen_ik14):,} unique inchikey14")

    # 2. Merge external COCONUT / PubChem natural product structures (Tier 2: Known DB)
    if external_path is not None and Path(external_path).exists():
        external_path = Path(external_path)
        if external_path.is_dir():
            external_path = external_path / "coconut_structures.parquet"
        precomputed = load_precomputed_standardization(external_path.with_name("coconut_standardized.parquet"))
        if precomputed is not None:
            ext_results, ext_name = precomputed, "coconut_standardized.parquet (precomputed)"
        else:
            if external_path.suffix == ".parquet":
                ext_smi = pd.read_parquet(external_path, columns=["smiles"])["smiles"].dropna().tolist()
            else:
                ext_smi = [line.split()[0] for line in external_path.read_text(encoding="utf-8").splitlines() if line.strip()]
            ext_results, ext_name = standardize_many(ext_smi, deadline=deadline), external_path.name
        n_before = len(seen_ik14)
        for res in ext_results:
            if res is not None:
                canon_smi, ik14, mass, formula = res
                if ik14 not in seen_ik14:
                    seen_ik14[ik14] = (canon_smi, mass, formula, 2)  # source=2 (external)
        ENV.log(f"Tier 2 ({ext_name}): +{len(seen_ik14) - n_before:,} new inchikey14")

    if not seen_ik14:
        raise RuntimeError("No candidate structures were standardized; check the train.parquet / COCONUT inputs.")

    records = [
        {
            "inchikey14": ik14,
            "canonical_smiles": vals[0],
            "exact_mass": float(vals[1]),  # float64: float32 rounds ~0.06 ppm at 1000 Da
            "molecular_formula": vals[2],
            "source_tier": np.uint8(vals[3]),
        }
        for ik14, vals in seen_ik14.items()
    ]

    catalog_df = pd.DataFrame(records)
    # Sort strictly by exact_mass ascending for O(log N) binary-search windowing
    catalog_df = catalog_df.sort_values("exact_mass", ascending=True).reset_index(drop=True)
    catalog_df["catalog_id"] = np.arange(len(catalog_df), dtype=np.int32)

    catalog_df.to_parquet(output_parquet_path, index=False, compression="zstd")
    print(f"Saved {len(catalog_df):,} unique InChIKey14 compounds to {output_parquet_path}")
    return catalog_df

## Cell 2: FP16 Memory-Mapped ChemFormer Encoding & FAISS Index Builder

Encoding into an on-disk `np.memmap` array in `float16` reduces storage by 50% (1M compounds × 256 dimensions × 2 bytes ≈ 512 MB) and allows Kaggle notebooks to map the embeddings directly from `/kaggle/input` without copying them into system RAM.

> **Review fix:** `torch.cuda.amp.autocast` is deprecated; `torch.autocast(device_type=...)` is used. Batches are length-sorted and trimmed to their longest SMILES, which leaves the masked `[CLS]` embedding unchanged and cuts CPU encoding cost ~6× (median SMILES is 39 of 256 tokens). This notebook runs on a CPU session, so it uses no GPU quota.

In [ ]:
# [Colab Companion Cell 2] Encode catalog into FP16 memory-mapped file + build FAISS & Formula indices
import faiss
import torch


@torch.no_grad()
def encode_catalog_to_memmap_and_faiss(
    catalog_df: pd.DataFrame,
    mol_encoder: torch.nn.Module,
    tokenizer,
    mmap_path: Path = IndexCFG.EMBEDDINGS_MMAP,
    faiss_path: Path = IndexCFG.FAISS_INDEX_PATH,
    formula_idx_path: Path = IndexCFG.FORMULA_INDEX_PATH,
    batch_size: int = IndexCFG.SHARD_BATCH_SIZE,
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu"),
) -> None:
    """
    Streams ChemFormer molecular embeddings directly to a float16 disk memmap,
    builds an inner-product FAISS index for global fallback search, and exports
    a CSR-style molecular formula lookup table for formula-constrained retrieval.
    """
    mol_encoder.eval().to(device)
    n_compounds = len(catalog_df)
    d_model = IndexCFG.D_MODEL

    # 1. Allocate on-disk float16 memory-mapped array: shape (N, 256)
    mmap_embs = np.memmap(
        mmap_path, dtype=np.float16, mode="w+", shape=(n_compounds, d_model)
    )

    smiles_list = catalog_df["canonical_smiles"].tolist()
    # Length-sorted batches trimmed to their longest SMILES: padding is masked in the encoder,
    # so the [CLS] embedding is unchanged while CPU cost drops ~6x (median SMILES is 39 of 256 tokens).
    order = np.argsort([len(s) for s in smiles_list], kind="stable")

    for b_i, start_idx in enumerate(range(0, n_compounds, batch_size)):
        rows = order[start_idx:start_idx + batch_size]
        token_Batch = torch.stack([tokenizer.encode(smiles_list[i], add_cls=True) for i in rows])
        used = int((token_Batch != tokenizer.pad_id).sum(dim=1).max())
        token_Batch = token_Batch[:, :used].to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
            _, e_m = mol_encoder(token_Batch)  # Normalized (B, 256)

        mmap_embs[rows] = e_m.float().cpu().numpy().astype(np.float16)

        if b_i % 50 == 0:
            mmap_embs.flush()
            ENV.log(f"Encoded {min(start_idx + batch_size, n_compounds):,} / {n_compounds:,} molecules...")

    mmap_embs.flush()

    # 2. Build FAISS Inner-Product Index (Cosine similarity on L2-normalized vectors)
    quantizer = faiss.IndexFlatIP(d_model)
    nlist = min(4096, max(1, int(math.sqrt(n_compounds))))
    if n_compounds >= 10000:
        index = faiss.IndexIVFFlat(quantizer, d_model, nlist, faiss.METRIC_INNER_PRODUCT)
        # Train IVF centroids on a representative sample
        sample_size = min(n_compounds, 131072)
        sample_idx = np.random.choice(n_compounds, size=sample_size, replace=False)
        train_vecs = np.asarray(mmap_embs[sample_idx], dtype=np.float32)
        index.train(train_vecs)
    else:
        index = quantizer

    # Stream shards into FAISS index
    for start_idx in range(0, n_compounds, batch_size * 16):
        end_idx = min(start_idx + batch_size * 16, n_compounds)
        shard_fp32 = np.asarray(mmap_embs[start_idx:end_idx], dtype=np.float32)
        index.add(shard_fp32)

    faiss.write_index(index, str(faiss_path))
    print(f"Saved FAISS index ({index.ntotal:,} vectors) to {faiss_path}")

    # 3. Build CSR-style inverted index for exact Molecular Formula retrieval (GLMR Formula Library)
    formula_groups = catalog_df.groupby("molecular_formula")["catalog_id"].apply(
        lambda s: np.asarray(s.values, dtype=np.int32)
    )
    formulas = np.array(list(formula_groups.index), dtype=str)
    lengths = np.array([len(arr) for arr in formula_groups.values], dtype=np.int32)
    offsets = np.zeros(len(lengths) + 1, dtype=np.int64)
    np.cumsum(lengths, out=offsets[1:])
    flat_indices = (
        np.concatenate(list(formula_groups.values))
        if len(lengths) > 0
        else np.empty(0, dtype=np.int32)
    )

    np.savez_compressed(
        formula_idx_path,
        formulas=formulas,
        offsets=offsets,
        flat_indices=flat_indices,
    )
    print(f"Saved {len(formulas):,} unique molecular formula buckets to {formula_idx_path}")

## Cell 3: Offline Wheel Bundler & Kaggle Dataset Uploader

Kaggle submissions run with internet disabled. This cell packages the sharded index, checkpoints, and pinned `rdkit==2026.03.3` wheels into a single versioned Kaggle Dataset.

> **Review fix:** `matchms` was bundled but never imported (and pulls in numba), and `pyarrow` ships with the Kaggle image. Wheels are downloaded with the running interpreter (`sys.executable -m pip`) so they match the image, and a failed download stops the run instead of producing an unusable offline dataset. Publishing happens from the repository with `tools/kaggle_pipeline.py publish-output nb2`.

In [ ]:
# [Colab Companion Cell 3] Bundle offline wheels + metadata and push to Kaggle Datasets
import subprocess
import sys

OFFLINE_WHEELS = ["rdkit==2026.03.3", "faiss-cpu"]  # matchms was never imported; pyarrow ships with Kaggle


def prepare_and_upload_kaggle_dataset(
    kaggle_username: str = ENV.KAGGLE_OWNER,
    dataset_slug: str = ENV.OUTPUT_SLUGS["nb2"],
    export_dir: Path = IndexCFG.EXPORT_DIR,
) -> None:
    """
    Downloads pinned wheels for offline installation and generates dataset-metadata.json
    for `kaggle datasets create` or `kaggle datasets version`.
    """
    wheels_dir = export_dir / "wheels"
    wheels_dir.mkdir(parents=True, exist_ok=True)

    # Wheels for this exact interpreter/platform (the Kaggle image that runs the submission)
    result = subprocess.run(
        [sys.executable, "-m", "pip", "download", "--dest", str(wheels_dir), "--only-binary=:all:"] + OFFLINE_WHEELS,
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(result.stdout[-2000:], result.stderr[-2000:])
        raise RuntimeError("pip download failed; offline submissions would have no RDKit/FAISS wheels.")
    ENV.log(f"Bundled wheels: {sorted(p.name for p in wheels_dir.glob('*.whl'))}")

    metadata = {
        "title": "CASMI 2026 GLMR Offline Embeddings and Wheels",
        "id": f"{kaggle_username}/{dataset_slug}",
        "licenses": [{"name": "CC-BY-NC-4.0"}],
    }
    meta_path = export_dir / "dataset-metadata.json"
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    print(f"Dataset staged at {export_dir}. After this run, publish it from the repository with:")
    print("  .venv/bin/python experiments/runs/RL77/tools/kaggle_pipeline.py fetch nb2")
    print("  .venv/bin/python experiments/runs/RL77/tools/kaggle_pipeline.py publish-output nb2")

## Cell 4: Zero-Copy Offline Retriever for Your Kaggle Submission Notebook

Drop this class into your Kaggle submission notebook (replacing `retrieve_top_k_candidates` in Cell 7 and Cell 8 of the main template). It reads embeddings directly from disk via `mode="r"` memory mapping and slices mass windows with `np.searchsorted` in under 5 milliseconds per molecule.

> **Review fix:** With fewer than K candidates in the window, the original returned only the FAISS hits and dropped the mass-matched candidates; FAISS also pads missing results with `-1`, which silently indexed the last memmap row. Constrained candidates are now kept first and FAISS fills the rest. Neutral mass uses `neutral_mass_from_precursor()` (formate/acetate spellings, `[2M+X]`), and `top_k`, `ppm_tol`, `in_library_boost`, and `nprobe` come from the Evidently weights.

In [ ]:
# [Kaggle Submission Cell] Zero-Copy Memory-Mapped Weight + Formula Pre-Retriever
class OfflineMassAndFormulaRetriever:
    """
    Zero-copy offline retriever for Kaggle submissions:
      1. Queries exact molecular formula bucket if a high-confidence formula is provided (MIST-CF / SIRIUS).
      2. Queries contiguous monoisotopic mass window [mass - tol, mass + tol] via O(log N) binary search.
      3. Fills the remaining slots from FAISS IVF search when fewer than K candidates match the window.
    """

    def __init__(
        self,
        dataset_dir: Optional[Path] = None,
        d_model: int = 256,
    ):
        # Resolved at call time: the notebook-2 output dataset is attached to notebooks 3-5 and the submission.
        dataset_dir = Path(dataset_dir) if dataset_dir else ENV.upstream_dir("nb2", "candidate_catalog_sorted.parquet")
        self.dataset_dir = dataset_dir
        self.catalog_df = pd.read_parquet(dataset_dir / "candidate_catalog_sorted.parquet")
        self.smiles_arr = self.catalog_df["canonical_smiles"].to_numpy()
        self.ik14_arr = self.catalog_df["inchikey14"].to_numpy()
        self.masses_arr = self.catalog_df["exact_mass"].to_numpy(dtype=np.float64)
        self.source_tier_arr = self.catalog_df["source_tier"].to_numpy(dtype=np.uint8)

        n_rows = len(self.catalog_df)
        self.mmap_embs = np.memmap(
            dataset_dir / "chemformer_embs_fp16.mmap",
            dtype=np.float16,
            mode="r",
            shape=(n_rows, d_model),
        )

        self.faiss_index = faiss.read_index(str(dataset_dir / "chemformer_ivfflat.index"))
        if hasattr(self.faiss_index, "nprobe"):
            self.faiss_index.nprobe = W.int("nb2.faiss.nprobe")

        # Load CSR formula index
        f_data = np.load(dataset_dir / "formula_offsets.npz", allow_pickle=False)
        formulas = f_data["formulas"]
        offsets = f_data["offsets"]
        flat_indices = f_data["flat_indices"]
        self.formula_to_span: Dict[str, Tuple[int, int]] = {
            str(f): (int(offsets[i]), int(offsets[i + 1])) for i, f in enumerate(formulas)
        }
        self.formula_flat_indices = flat_indices

    def window(self, est_neutral_mass: float, ppm_tol: float) -> Tuple[int, int]:
        tol_da = max(est_neutral_mass * (ppm_tol * 1e-6), W("nb2.retrieval.min_tolerance_da"))
        left_idx = int(np.searchsorted(self.masses_arr, est_neutral_mass - tol_da, side="left"))
        right_idx = int(np.searchsorted(self.masses_arr, est_neutral_mass + tol_da, side="right"))
        return left_idx, right_idx

    def get_candidate_indices(
        self,
        est_neutral_mass: float,
        ppm_tol: float = W("nb2.retrieval.ppm_tol"),
        predicted_formulas: Optional[List[str]] = None,
    ) -> np.ndarray:
        """Returns catalog row indices constrained by formula priors and/or ppm mass window."""
        matched_indices = []

        # Priority A: Formula-based library candidates (GLMR Table 1 & 2)
        if predicted_formulas:
            for form in predicted_formulas:
                span = self.formula_to_span.get(form)
                if span is not None:
                    matched_indices.append(self.formula_flat_indices[span[0] : span[1]])

        # Priority B: Weight-based library candidates via O(log N) binary search
        left_idx, right_idx = self.window(est_neutral_mass, ppm_tol)
        if right_idx > left_idx:
            matched_indices.append(np.arange(left_idx, right_idx, dtype=np.int32))

        if not matched_indices:
            return np.empty(0, dtype=np.int32)
        return np.unique(np.concatenate(matched_indices)).astype(np.int64)

    @torch.no_grad()
    def retrieve_top_k_batch(
        self,
        e_s_batch: torch.Tensor,
        precursor_mzs: torch.Tensor,
        adducts: List[str],
        adduct_mass_shift: Optional[Dict[str, float]] = None,
        top_k: int = W.int("nb2.retrieval.top_k"),
        ppm_tol: float = W("nb2.retrieval.ppm_tol"),
        in_library_boost: float = W("nb2.retrieval.in_library_boost"),
        formula_priors_batch: Optional[List[List[str]]] = None,
        ionization_modes: Optional[List[str]] = None,
    ) -> Tuple[torch.Tensor, List[List[int]]]:
        """
        Retrieves top-K candidate embeddings H^m_K per molecule directly from the FP16 memmap.
        Applies a small prior boost to Tier-1 in-library compounds when spectral similarity ties.
        `adduct_mass_shift` is accepted for the PDF call signature; neutral mass now comes from
        neutral_mass_from_precursor(), which also handles formate/acetate spellings and [2M+X] multimers.
        """
        device = e_s_batch.device
        e_s_np = e_s_batch.detach().cpu().numpy().astype(np.float32)
        top_k = min(top_k, len(self.masses_arr))

        batch_embs = []
        batch_cat_ids = []

        for b in range(len(e_s_np)):
            prec_mz = float(precursor_mzs[b].item())
            mode = ionization_modes[b] if ionization_modes is not None else "positive"
            neutral_mass = neutral_mass_from_precursor(prec_mz, adducts[b], mode)
            f_priors = formula_priors_batch[b] if formula_priors_batch else None

            cand_ids = self.get_candidate_indices(neutral_mass, ppm_tol, f_priors)
            cand_vecs = np.asarray(self.mmap_embs[cand_ids], dtype=np.float32) if len(cand_ids) else np.empty((0, e_s_np.shape[1]), np.float32)
            scores = cand_vecs @ e_s_np[b]
            # Add slight tie-breaking prior for known training library compounds (source_tier == 1)
            tier1_mask = (self.source_tier_arr[cand_ids] == 1).astype(np.float32)
            scores = scores + in_library_boost * tier1_mask

            if len(cand_ids) >= top_k:
                # Slice directly from disk memmap without loading full array into RAM
                top_local = np.argpartition(-scores, kth=top_k - 1)[:top_k]
                top_local = top_local[np.argsort(-scores[top_local])]
                chosen_ids = cand_ids[top_local]
            else:
                # Keep every constrained candidate, then fill from global FAISS inner-product search.
                # (The original replaced the constrained candidates with the FAISS hits, and FAISS pads
                # missing results with -1, which silently indexed the last memmap row.)
                ranked = cand_ids[np.argsort(-scores)]
                _, faiss_ids = self.faiss_index.search(e_s_np[b : b + 1], top_k + len(ranked))
                taken = set(ranked.tolist())
                fill = [int(i) for i in faiss_ids[0] if i >= 0 and int(i) not in taken][: top_k - len(ranked)]
                chosen_ids = np.concatenate([ranked, np.asarray(fill, dtype=np.int64)])
                if len(chosen_ids) < top_k:  # tiny catalogs only
                    chosen_ids = np.resize(chosen_ids, top_k)
            chosen_vecs = np.asarray(self.mmap_embs[chosen_ids], dtype=np.float32)

            batch_cat_ids.append(chosen_ids.astype(np.int64).tolist())
            batch_embs.append(torch.from_numpy(chosen_vecs).to(device))

        return torch.stack(batch_embs, dim=0), batch_cat_ids

    def in_window(self, cat_ids: List[int], neutral_mass: float, ppm_tol: float = W("nb2.retrieval.ppm_tol")) -> np.ndarray:
        left_idx, right_idx = self.window(neutral_mass, ppm_tol)
        ids = np.asarray(cat_ids)
        return (ids >= left_idx) & (ids < right_idx)

## Cell 5: Build the Store on the Competition Data and Measure Retrieval

The PDF stops at the retriever class. This cell runs Cells 1–4 for real: the notebook-1 molecule encoder embeds every candidate, and retrieval is scored on notebook 1's held-out timsTOF molecules (MRR@25, recall@K, and whether the true structure is in the mass window at all). Outputs in `/kaggle/working` become the `casmi26-rl77-nb2-store-outputs` dataset that notebooks 3–5 and the offline submission attach.

`in_library_boost` is **not** fitted here: every validation molecule comes from `train.parquet` (tier 1), so a validation sweep would always push the boost up while the hidden test set also contains out-of-library structures.

In [ ]:
# [Colab Companion Cell 5] Build the offline store from the competition data, measure retrieval, save outputs
@torch.no_grad()
def evaluate_store_retrieval(retriever: "OfflineMassAndFormulaRetriever", model, loader, answer_key: Dict[str, str],
                             formula_priors_fn=None) -> Dict[str, float]:
    """Stage 1 retrieval from the offline store on held-out molecules: MRR@25, recall@K, mass-window coverage."""
    model.eval()
    preds, gt, hit_k, in_catalog_window = {}, {}, [], []
    for batch in loader:
        batch = {k: (v.to(CFG.DEVICE) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}
        _, e_s_mol, _ = model.encode_molecule_spectra(batch)
        priors = formula_priors_fn(batch) if formula_priors_fn else None
        _, ids_batch = retriever.retrieve_top_k_batch(
            e_s_mol, batch["precursor_mz"], batch["primary_adduct"],
            formula_priors_batch=priors, ionization_modes=batch["ionization_mode"],
        )
        e_np = e_s_mol.cpu().numpy().astype(np.float32)
        for b, mol_id in enumerate(batch["molecule_id"]):
            ids = ids_batch[b]
            neutral = neutral_mass_from_precursor(
                float(batch["precursor_mz"][b]), batch["primary_adduct"][b], batch["ionization_mode"][b]
            )
            rec = {
                "cand_smiles": retriever.smiles_arr[ids].tolist(),
                "stage1_scores": np.asarray(retriever.mmap_embs[ids], dtype=np.float32) @ e_np[b],
                "stage2_scores": None,
                "in_window": retriever.in_window(ids, neutral),
                "gen_smiles": None, "gen_ik14": None, "gen_mass_ok": False, "backfill_smiles": [],
            }
            truth = answer_key[str(mol_id)]
            preds[mol_id], gt[mol_id] = assemble_ranked_candidates(rec, 0.0, 0), truth
            hit_k.append(truth in set(retriever.ik14_arr[ids].tolist()))
            lo, hi = retriever.window(neutral, W("nb2.retrieval.ppm_tol"))
            in_catalog_window.append(truth in set(retriever.ik14_arr[lo:hi].tolist()))
    return {
        "mrr25": compute_casmi_mrr25(preds, gt),
        "recall_at_k": float(np.mean(hit_k)),
        "truth_in_mass_window": float(np.mean(in_catalog_window)),
        "n": len(gt),
    }


def run_offline_store_pipeline() -> Dict:
    out_dir = IndexCFG.EXPORT_DIR
    metrics: Dict = {"weights_version": W.version}
    nb1_dir = ENV.upstream_dir("nb1", "glmr_model.pt")
    model, tokenizer = load_glmr_checkpoint(nb1_dir / "glmr_model.pt", device=CFG.DEVICE)

    # 1-2. Catalog (train.parquet tier 1 + COCONUT tier 2), FP16 memmap, FAISS and formula indices
    catalog_df = build_mass_sorted_catalog()
    metrics["catalog"] = {
        "n_structures": int(len(catalog_df)),
        "n_tier1_train": int((catalog_df["source_tier"] == 1).sum()),
        "n_tier2_external": int((catalog_df["source_tier"] == 2).sum()),
        "n_formulas": int(catalog_df["molecular_formula"].nunique()),
    }
    ENV.log(f"Catalog: {metrics['catalog']}")
    encode_catalog_to_memmap_and_faiss(catalog_df, model.mol_encoder, tokenizer, device=CFG.DEVICE)

    # 3. Offline wheels for internet-off submissions
    prepare_and_upload_kaggle_dataset()

    # 4. Retrieval quality on notebook 1's held-out timsTOF molecules (same split everywhere).
    # Every validation molecule is a train.parquet structure (tier 1), so this measures in-library retrieval.
    retriever = OfflineMassAndFormulaRetriever(out_dir)
    split_dir = ENV.upstream_dir("nb1", "val_inchikey14.txt")
    val_df = load_spectra_for_inchikeys(CFG.DATA_DIR, read_split(split_dir, "val"), timstof_window_only=True)
    metrics["val_store_stage1"] = evaluate_store_retrieval(
        retriever, model, make_loader(val_df, tokenizer, train=False, batch_size=32), ground_truth_ik14(val_df)
    )
    ENV.log(f"Validation retrieval from the offline store: {metrics['val_store_stage1']}")

    ENV.write_json(out_dir / "metrics.json", metrics)
    ENV.write_run_manifest("nb2", out_dir, W.version, {"metrics": metrics, "nb1_outputs": str(nb1_dir)})
    ENV.log(f"Notebook 2 outputs written to {out_dir}")
    return metrics


if __name__ == "__main__":
    NB2_METRICS = run_offline_store_pipeline()

## Memory & Latency Profile on Kaggle

| Resource Metric | Unsharded In-Memory Tensor | Sharded FP16 Memmap + Binary Search |
| --- | --- | --- |
| **1.5M Compound Disk Size** | 1.54 GB (`float32`) | 768 MB (`float16 .mmap`) |
| **Kaggle Peak System RAM** | Over 6.5 GB (risks OOM crash) | Under 450 MB (OS page-cache backed) |
| **Mass-Window Slice Latency** | 45 ms per molecule (boolean mask) | Under 1.2 ms per molecule (`searchsorted`) |
| **Total 400-Molecule Retrieval** | ~18 seconds | Under 1 second (plus ~7 min Beam Search) |

The table above shows how sorting by `exact_mass` and mapping `float16` vectors keeps total Kaggle submission runtime under 8 minutes while leaving over 12 GB of system RAM free for PyTorch beam search and ensemble blending. *(Figures are the PDF's estimates; `metrics.json` records the measured run.)*